# Partner tasks: does fusion help when the second input is a partner?

Every result in this project so far asks one question: does reading a second view of **the
same molecule** help predict a property of it? The stability benchmarks give a transcript and
let three encoders read it as DNA, as RNA and as protein, and fusion is the attempt to combine
those views. Three seeds settled what that is worth, and the answer was narrow: three gains
hold, one is unresolved, and most of the 96 comparisons are unresolved either way.

This notebook asks the other question, which is the one Mina's plan calls the task-specific
multimodal setting: does fusion help when the two inputs are **two different molecules that
interact**? A pair task has no codon grid, no shared coordinate, and no possibility of one
encoder standing in for the other. The label is a property of the pair, so a model that cannot
read both sides cannot be right for the right reason.

Three pairings, five tables:

| Task | Table | Pairs | Label |
| --- | --- | --- | --- |
| RNA and RNA-binding protein | `rpi2241` | 4,479 | binds / does not bind |
| RNA and RNA-binding protein | `npinter-matched` | 22,516 | binds / does not bind |
| DNA and binding protein | `dapseq` | 15,905 | binds / does not bind |
| Protein and ligand | `davis` | 25,772 | measured pKd |
| Protein and ligand | `kiba` | 117,657 | measured KIBA score |

The two controls matter more here than anywhere else in the project. A pair table can be
predicted surprisingly well by **how popular each entity is**, without reading a single
nucleotide: a protein that binds many things probably binds this one too. `degree` measures
exactly that, and an arm that does not clear it has shown nothing. `partner_reassigned` keeps
each real anchor and gives it a partner nobody measured; a model that reads the pair should
fall on those rows, and a model that has only learned popularity should not notice.

---

## Setup

Use the [repository environment setup](../../README.md#setup). The pair tables must already be
built under `Code/stage4/data/pairs/`; see `shared_code.fusion.pair_sources` and
`shared_code.fusion.pair_dna` for how each was constructed. Entity embedding extraction is the
only stage that loads pretrained encoder weights, and it is disabled by default.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import torch

In [2]:
# Resolve the checkout from a repository, Code, or notebook working directory.
start = Path.cwd().resolve()
REPO_DIR = next((path for path in (start, *start.parents)
                 if (path / "Code" / "shared_code").is_dir()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Open this notebook from the repository or one of its subdirectories.")
CODE_DIR = REPO_DIR / "Code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from shared_code.fusion import (entity_embeddings, pair_sources, pair_study, pair_tasks,
                                pair_training, run_identity)

### Settings and execution boundaries

Six arms per split, three splits, one seed: **18 fits per table**. Two single-partner
baselines establish what one side alone can do; the four fusion arms are what has to beat
them. `cross_attention` is the gated cross-attention design Mina asked for, and
`interleaved_mamba` is the Coupled Mamba design applied to a pair instead of to two views of
one sequence. The three gate variants and the reversed Mamba order are optional and are not
part of the 18.

The three splits are the point of the design, not a robustness check. Holding out **left**
entities asks whether the model generalises to an unseen RNA or DNA region; holding out
**right** entities asks whether it generalises to an unseen protein or ligand; holding out
**both** asks the hard question, and is the only one of the three that a method claiming to
read the interaction should be judged on.

| Setting | Basis |
| --- | --- |
| Entity-group-disjoint splits, 70/15/15, split seed 42 | Agreed project grouping rule. Groups are exact sequence identity, and for `dapseq` a 10 kb genomic bin on the left and a factor family on the right. A pair whose two entities fall in different partitions is dropped, and the count is recorded |
| One encoder per side, the modality's default | `entity_embeddings.MODALITY_DEFAULTS`: RiNALMo for RNA and HyenaDNA for DNA, which are the encoders the three-seed cohort found the stability gains under; ESM-2 150M for protein; ChemBERTa for ligand |
| 1,024-token cap per side | Padded pair batches, not a biological choice. Every entity at the cap is named in the cache metadata and counted in the results |
| Selection on validation AUPRC (binding) or Spearman (affinity) | A binding table is imbalanced, so AUPRC is read rather than AUROC. Test rows are scored once, with the weights validation chose |
| Grouped paired bootstrap, 2,000 draws, grouped by the right side | The project's comparison rule, reused through `train.seed_averaged_bootstrap` |
| 8 pairs through the model at once, accumulated to a batch of 32 | Memory, not training. Cross-attention between two 1,024-token streams holds an attention matrix per head per direction, which at batch 32 exhausted a laptop's GPU. Each batch is still one optimizer step on the whole batch's gradient |

### What the three splits cost, measured

A dual-disjoint split on a dense table is expensive, and the cost is arithmetic rather than a
choice: holding out 15% of left groups and 15% of right groups leaves roughly 0.15 x 0.15 of
the grid in the test cell, and every pair whose two entities land in different partitions has
to be dropped. Measured at split seed 42:

| Table | left out: test rows | right out: test rows | both out: test rows | both: dropped |
| --- | --- | --- | --- | --- |
| `rpi2241` | 894 | 688 | 132 | 2,201 of 4,479 |
| `npinter-matched` | 3,434 | 6,295 | 351 | 8,445 of 22,516 |
| `dapseq` | 2,359 | 2,385 | 376 | 7,189 of 15,905 |
| `davis` | 3,876 | 3,790 | 570 | 11,912 of 25,772 |
| `kiba` | 14,961 | 18,315 | 2,088 | 52,625 of 117,657 |

Three consequences, all of which change how a result is read:

**The degree baseline is 0.500 on every "both" split, by construction.** When neither entity
of a test pair was seen in training, the baseline has no information about either one and
predicts the same value for every row. A constant prediction scores 0.5 on a binding table and
has no Spearman at all on an affinity table. This is the design working: the popularity
shortcut is unavailable on the split that matters. It also means the baseline is only
informative on the single-side splits, and the notebook reports it as undefined rather than as
a number to clear.

**On `rpi2241` with proteins held out, the degree baseline reaches 0.867 AUROC.** Unseen
proteins against seen RNAs is very nearly a popularity contest: knowing how many partners an
RNA has recorded predicts the label at 0.867 without reading a base. Any arm on that split has
to clear 0.867, not 0.5, before it has shown anything.

**`npinter-matched` has a degree baseline below chance** (0.249 with proteins held out). The
negatives were sampled with degree matching, which inverted the popularity signal rather than
removing it. The matched table is therefore not a neutral control, and `rpi2241`'s supplied
negatives and `npinter-matched`'s sampled ones should be read as two different tasks.

**Known gaps, recorded rather than fixed.** Groups are exact-identity only: a protein 95%
identical to a training protein counts as unseen, which makes every number here optimistic.
`dapseq` has only 15 factor families in total, so a "both" split leaves 4 of them in test and
any claim about unseen factors rests on those 4. `davis` has 71% of its values at the pKd floor
of 5.0, which is a censoring limit and not a measurement, so Spearman on that table largely
measures separation of the floor from everything else. `dapseq` is in vitro, so it measures
what a factor can bind, not what it occupies in a cell. One seed is not three; the seed cohort
comes after a first table is reviewed.

In [3]:
TABLE = "rpi2241"  # One of: rpi2241, npinter-matched, dapseq, davis, kiba.
HELD_OUT = "left"  # The split the analysis cells below report on.
SPLIT_SEED = 42
SEED = 0
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_SEED = 0
MAX_TOKENS = 1024
MICROBATCH = 8   # Pairs put through the model at once. A memory setting, not a training one:
                 # a batch is still one optimizer step. Lower it if the GPU runs out of memory.
# Whether a batch is drawn from rows of similar length rather than at random. A batch is
# padded to its longest member, and this table holds a few entities at the token cap among
# many short ones, so a random batch is nearly always padded to the cap and almost all of
# the memory goes on padding. This changes which rows share a batch, so unlike MICROBATCH
# it changes the fit, and it is recorded in the identity. Off until the arms already
# saved for this table are refitted under it, and it does not on its own lower the peak:
# the batch holding the capped entities is still as wide as it is today.
BUCKET_BY_LENGTH = False
ENCODERS = None  # None takes each modality's default; {"left": "rnafm"} overrides one side.

READ_SAVED_RESULTS = True
RUN_TABLE_AUDIT = False      # Read the table and the split; load no encoder, fit nothing.
RUN_EXTRACTION = False       # Embed both sides' entities. Loads pretrained weights.
RUN_TRAINING = False         # Off: the CPU cell further down does tonight's fitting instead.
RUN_ANALYSIS = True          # Opens the caches, and reports the saved arms at the end.
INCLUDE_OPTIONAL = False     # Gate variants and the reversed Mamba order.
KEYS = None                  # None selects the declared panel; a list selects arms.

DATA_ROOT = CODE_DIR / "stage4" / "data" / "pairs"
CACHE_ROOT = CODE_DIR / "stage4" / "cache" / "pair_entities"
RESULTS_ROOT = CODE_DIR / "stage4" / "results" / "partner_tasks"
DEVICE = "cpu"               # The attention arms do not fit in this machine's Metal budget.

CONFIG = pair_training.PairConfig(seed=SEED, max_tokens=MAX_TOKENS,
                                  microbatch=MICROBATCH,
                                  bucket_by_length=BUCKET_BY_LENGTH)


In [4]:
actions = [RUN_TABLE_AUDIT, RUN_EXTRACTION, RUN_TRAINING, RUN_ANALYSIS]
if sum(actions) > 1:
    raise ValueError("Choose one result-producing action per execution.")
if READ_SAVED_RESULTS and (RUN_EXTRACTION or RUN_TRAINING):
    raise ValueError("Disable READ_SAVED_RESULTS explicitly for extraction or training.")
if HELD_OUT not in pair_tasks.HELD_OUT:
    raise ValueError(f"HELD_OUT must be one of {pair_tasks.HELD_OUT}")
if not (DATA_ROOT / TABLE / "dataset.json").exists():
    raise FileNotFoundError(
        f"{DATA_ROOT / TABLE} holds no built table. Build it first: "
        "python -m shared_code.fusion.pair_sources <RPIMotif/Data> <out> for the RNA tables, "
        "or shared_code.fusion.pair_dna.main(...) for dapseq.")
print("device:", DEVICE)
print("table:", DATA_ROOT / TABLE)
print("results:", RESULTS_ROOT / TABLE / f"held_out_{HELD_OUT}")

device: cpu
table: /Users/julianmckinley/Desktop/Masters/CapstoneCourse/Mina/Code/stage4/data/pairs/rpi2241
results: /Users/julianmckinley/Desktop/Masters/CapstoneCourse/Mina/Code/stage4/results/partner_tasks/rpi2241/held_out_left


## The table, and what the split does to it

`read_pair_dataset` restores the built table: the pairs, each side's strings, each side's
groups, and what each side is. The split is computed here in every mode, because it is
deterministic from the table and the seed, and every saved fit records the row indices it was
computed from so a later read can tell whether it is looking at the same partition.

The two numbers to read before anything else are **how many pairs were dropped** to keep the
partitions disjoint, and **what the label balance is in each partition**. A split that drops
most of a dense table is not a split, and a test partition with almost no positives cannot
support an AUPRC.

In [5]:
frame = pair_sources.read_pair_dataset(DATA_ROOT / TABLE)
metadata = json.loads((DATA_ROOT / TABLE / "dataset.json").read_text())

splits = {held_out: pair_tasks.disjoint_split(frame, held_out=held_out, seed=SPLIT_SEED)
          for held_out in pair_tasks.HELD_OUT}
split = splits[HELD_OUT]

print(f"{frame.name}: {len(frame.pairs)} pairs, kind={frame.kind}")
print(f"  left  ({frame.modality('left')}): {len(frame.left_sequences)} entities, "
      f"{len(set(frame.groups('left')))} groups among the pairs")
print(f"  right ({frame.modality('right')}): {len(frame.right_sequences)} entities, "
      f"{len(set(frame.groups('right')))} groups among the pairs")
print(f"  grouping coverage: {frame.grouping_coverage()}")
print(f"  note: {frame.note}")

display(pd.DataFrame([
    {"held_out": held_out, "dropped": s["dropped"],
     **{name: len(s["indices"][name]) for name in pair_tasks.SPLITS},
     **{f"{name}_positive": (int(np.sum(frame.pairs.iloc[s["indices"][name]]["label"] == 1))
                             if frame.kind == "binding" else None)
        for name in pair_tasks.SPLITS}}
    for held_out, s in splits.items()]).set_index("held_out"))

rpi2241: 4479 pairs, kind=binding
  left  (rna): 968 entities, 440 groups among the pairs
  right (protein): 3582 entities, 951 groups among the pairs
  grouping coverage: {'left': {'entities': 968, 'grouped': 968}, 'right': {'entities': 3582, 'grouped': 3582}}
  note: Positives are RNA and protein chains of one structural complex. The supplied negatives pair a protein from one complex with an RNA from another: nobody measured them, and they are predictable from how often each chain appears.


,dropped,train,val,test,train_positive,val_positive,test_positive
held_out,,,,,,,
left,0,2891,694,894,1381,395,462
right,0,3079,712,688,1446,399,393
both,2201,2026,120,132,953,76,79


### Does this table support the comparison at all?

`RUN_TABLE_AUDIT` reads the table and the three splits and reports what they hold, without
loading an encoder or fitting anything. It is the cheap check to run before committing a GPU:
the label provenance per partition (how many labels are measured and how many were sampled),
the degree baseline on the test rows of each split, and the length distribution of both sides
against the token cap.

The degree baseline here is the number that decides whether the table is worth running. It
reads no sequence. If it already scores 0.9 AUROC on the test rows, the table is a popularity
contest and a fusion result on it would mean nothing.

In [6]:
if RUN_TABLE_AUDIT:
    rows = []
    for held_out, s in splits.items():
        degree = pair_tasks.degree_baseline(frame, s["indices"]["train"], s["indices"]["test"])
        truth = frame.pairs.iloc[s["indices"]["test"]]["label"].to_numpy(
            dtype=np.int64 if frame.kind == "binding" else np.float64)
        rows.append({"held_out": held_out,
                     **pair_tasks.scores_for(frame.kind, truth, degree)})
    print("Degree baseline on the test rows. It reads no sequence.")
    print("On the 'both' split it has no information about either entity, so a constant "
          "prediction is expected: 0.5 on a binding table, undefined on an affinity one.")
    display(pd.DataFrame(rows).set_index("held_out"))

    print("\nLabel provenance per partition")
    display(pd.DataFrame({f"{held_out}/{name}": s["provenance"][name]
                          for held_out, s in splits.items()
                          for name in pair_tasks.SPLITS}).fillna(0).astype(int).T)

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
    for axis, side in zip(axes, pair_tasks.SIDES):
        lengths = np.array([len(s) for s in frame.sequences(side).values()])
        axis.hist(lengths, bins=40, color="#444444")
        axis.axvline(MAX_TOKENS, color="#b00020", linestyle="--",
                     label=f"{MAX_TOKENS}-token cap")
        over = int(np.sum(lengths > MAX_TOKENS))
        axis.set_title(f"{side} ({frame.modality(side)}): {over} of {len(lengths)} over the cap")
        axis.set_xlabel("characters")
        axis.legend()
    fig.tight_layout()
    plt.show()

## Embed each entity once

This is the step the stability benchmarks did not need. There, one row was one molecule read
three ways, so the cache was keyed by row. Here a protein appears in hundreds of pairs, and
embedding it per pair would embed it hundreds of times. `entity_embeddings` caches one
embedding per entity, keyed by name, with no codon grid, and records every entity that reached
the token cap.

Extraction is resumable: shards are written atomically and validated on a later run, so an
interrupted job continues where it stopped. A cache built from different inputs or a different
encoder is reported and left alone rather than overwritten.

In [7]:
needs_embeddings = RUN_EXTRACTION or RUN_TRAINING or RUN_ANALYSIS
embeddings = summaries = None
if RUN_EXTRACTION:
    embeddings, summaries = entity_embeddings.build(
        frame, CACHE_ROOT, DEVICE, encoders=ENCODERS, max_tokens=MAX_TOKENS)
elif needs_embeddings:
    # Open the caches without an encoder; raises if a side has not been extracted yet.
    embeddings, summaries = {}, {}
    for side in pair_tasks.SIDES:
        encoder = entity_embeddings.encoder_for(frame.modality(side),
                                                (ENCODERS or {}).get(side))
        directory = entity_embeddings.cache_dir_for(CACHE_ROOT, frame, side, encoder.key)
        if not (directory / "entities.json").exists():
            raise FileNotFoundError(
                f"{directory} holds no entity cache. Run this notebook once with "
                "RUN_EXTRACTION = True and READ_SAVED_RESULTS = False.")
        summaries[side] = entity_embeddings.summarize(directory)
        embeddings[side] = entity_embeddings.load(directory)

if summaries is not None:
    display(pd.DataFrame(summaries).T[
        ["modality", "encoder", "entities", "width", "token_limit", "at_token_limit",
         "at_token_limit_share"]])
    print(pair_study.token_limit_note(summaries))

,modality,encoder,entities,width,token_limit,at_token_limit,at_token_limit_share
left,rna,rinalmo,968,640,1022,263,0.2717
right,protein,esm2_150m,3582,640,1022,10,0.0028


rna through rinalmo: 263 of 968 entities (27.2%) reached the 1022-token limit; protein through esm2_150m: 10 of 3582 entities (0.3%) reached the 1022-token limit


## Read the saved evidence first

A fit is identified by the table, the split's row indices, both token caches, the
architecture, the optimizer settings and the seed. The reader reports a missing or
incompatible record rather than replacing it.

In [8]:
if summaries is not None:
    inventory = pd.DataFrame(pair_study.saved_inventory(
        RESULTS_ROOT, frame, CONFIG, summaries, splits,
        include_optional=INCLUDE_OPTIONAL))
    display(inventory.groupby(["held_out", "status"]).size().rename("fits").to_frame())
    if (inventory["status"] == "incompatible").any():
        display(inventory.loc[inventory["status"] == "incompatible", ["key", "held_out", "issue"]])
    saved = inventory[inventory["status"] == "saved"]
    if len(saved):
        display(saved.set_index(["held_out", "key"]).filter(like="test_"))

fits
held_out status       
both     saved       6
left     missing     2
         saved       4
right    missing     6

test_auroc  test_auprc  test_positives  test_rows  \
held_out key                                                                    
left     left_only            0.696464    0.669097           462.0      894.0   
         right_only           0.620576    0.613977           462.0      894.0   
         concat               0.825462    0.785739           462.0      894.0   
         bilinear             0.837061    0.820414           462.0      894.0   
both     left_only            0.557440    0.649471            79.0      132.0   
         right_only           0.558753    0.618620            79.0      132.0   
         concat               0.657989    0.688514            79.0      132.0   
         bilinear             0.658228    0.686148            79.0      132.0   
         cross_attention      0.675185    0.704501            79.0      132.0   
         interleaved_mamba    0.642942    0.682740            79.0      132.0   

                            test_positive_rate  
held_out key                                    
left     left_only                    0.516779  
         right_only                   0.516779  
         concat                       0.516779  
         bilinear                     0.516779  
both     left_only                    0.598485  
         right_only                   0.598485  
         concat                       0.598485  
         bilinear                     0.598485  
         cross_attention              0.598485  
         interleaved_mamba            0.598485

## Fit the panel

Each arm trains on the training rows, selects its epoch on validation, and scores the test
rows once. An arm whose evidence is already on disk under this identity is read rather than
refitted, so an interrupted run continues cheaply.

Both controls are scored for each arm as it finishes: `degree`, which reads no sequence, and
`partner_reassigned`, which keeps each real anchor and gives it an unmeasured partner. The
second is only defined for a binding table, since an invented pair has no measured affinity.

In [9]:
panel = None
if RUN_TRAINING:
    producer = run_identity.producer_record(device=DEVICE, repository=REPO_DIR)
    panel = pair_study.run_panel(frame, split, embeddings, CONFIG, summaries, RESULTS_ROOT,
                                 device=DEVICE, keys=KEYS,
                                 include_optional=INCLUDE_OPTIONAL, producer=producer)
    print(f"\nfitted or reused {len(panel['results'])} arms; reused {len(panel['reused'])}")

## Fitting on the CPU

`cross_attention` and `interleaved_mamba` exhaust this machine's Metal memory budget on this table: a quarter of its RNA entities sit at the 1,022-token cap, so nearly every batch is padded to that width and the attention matrix does not fit. This cell fits them on the CPU instead, reusing the arms already saved.


In [ ]:
# Finish rpi2241 on the CPU.
# The attention arms do not fit in this machine's Metal budget, which is nearly full before
# the fit starts. The CPU has the whole machine's RAM and no 30 GiB cap: slower, and it
# finishes. Arms already saved are reused rather than refitted, so this fits the two missing
# left-held-out arms and then the full right-held-out panel. Every result records
# producer.device, so a CPU-fitted arm stays auditable against an MPS-fitted one.
import time

for held_out in ("left", "right"):
    print("\n" + "=" * 70)
    print(f"{held_out} held out, on cpu, started {time.strftime('%H:%M:%S')}")
    print("=" * 70)
    producer = run_identity.producer_record(device="cpu", repository=REPO_DIR)
    panel = pair_study.run_panel(frame, splits[held_out], embeddings, CONFIG, summaries,
                                 RESULTS_ROOT, device="cpu", keys=None,
                                 include_optional=INCLUDE_OPTIONAL, producer=producer)
    print(f"{held_out}: fitted or reused {len(panel['results'])} arms; "
          f"reused {len(panel['reused'])}; finished {time.strftime('%H:%M:%S')}")

print("\nBoth splits saved. The analysis cells below report on HELD_OUT.")



left held out, on cpu, started 19:59:25

rpi2241 [left held out] cross_attention
cross_attention epoch 1: loss 0.5564, val auprc 0.8588
cross_attention epoch 2: loss 0.4251, val auprc 0.8717
cross_attention epoch 3: loss 0.3707, val auprc 0.8920
cross_attention epoch 4: loss 0.3123, val auprc 0.8617
cross_attention epoch 5: loss 0.2674, val auprc 0.8590
cross_attention epoch 6: loss 0.2261, val auprc 0.9023
cross_attention epoch 7: loss 0.2152, val auprc 0.9111
cross_attention epoch 8: loss 0.1741, val auprc 0.9100
cross_attention epoch 9: loss 0.1607, val auprc 0.8881
cross_attention epoch 10: loss 0.1407, val auprc 0.8768
cross_attention epoch 11: loss 0.1440, val auprc 0.8677
cross_attention epoch 12: loss 0.1151, val auprc 0.8961

rpi2241 [left held out] interleaved_mamba
interleaved_mamba epoch 1: loss 0.5668, val auprc 0.8398
interleaved_mamba epoch 2: loss 0.4830, val auprc 0.8321
interleaved_mamba epoch 3: loss 0.4256, val auprc 0.8689
interleaved_mamba epoch 4: loss 0.3987, v

## What the arms are worth

Three readings, in this order.

**Against the degree baseline.** An arm that does not clear the baseline that reads no
sequence has not shown it reads the pair, whatever its absolute score.

**Against the single-partner baselines.** `comparison_rows` compares every fusion arm with the
better of `left_only` and `right_only`, paired on the same test rows, with whole entity groups
resampled together. An interval that contains zero is unresolved, and is reported as unresolved
rather than as a tie or as evidence of no effect.

**Against the reassigned partner.** The gap between a model's mean score on real positive pairs
and on the same anchors given unmeasured partners. A model that reads the interaction should
show a large gap. A small gap on an arm with a high AUROC is the warning sign: it scored well
without the partner mattering.

In [ ]:
if RUN_ANALYSIS:
    panel = {"results": {}}
    for key in pair_study.panel_keys(include_optional=INCLUDE_OPTIONAL):
        path = pair_study.result_path(RESULTS_ROOT, frame, key, CONFIG, held_out=HELD_OUT)
        identity = pair_study.study_identity(frame, split, key, CONFIG, summaries)
        payload, issue = pair_study.read_result(path, identity)
        if issue is None:
            panel["results"][key] = payload
        else:
            print(f"  {key}: {issue}")

if panel and panel["results"]:
    display(pd.DataFrame(pair_study.panel_table(panel, frame)).set_index("key"))
    gap = pair_study.degree_gap(panel, frame)
    if gap and gap["degree_defined"]:
        print(f"\nBest arm {gap['best_arm']}: {gap['metric']} {gap['best']:.4f} against a "
              f"degree baseline of {gap['degree']:.4f} (gap {gap['gap']:+.4f})")
    elif gap:
        print(f"\nBest arm {gap['best_arm']}: {gap['metric']} {gap['best']:.4f}. The degree "
              "baseline is undefined on this split: neither entity of a test pair was seen in "
              "training, so it predicts one value for every row. That is the split working, "
              "and it means there is no popularity number to clear here.")

In [ ]:
comparison = None
if panel and len(panel["results"]) > 1:
    comparison = pair_training.comparison_rows(
        pair_study.results_for_comparison(panel), frame, split,
        groups="right", draws=BOOTSTRAP_DRAWS, seed=BOOTSTRAP_SEED)
    table = pd.DataFrame(comparison["rows"])
    # The project's three verdicts. An interval containing zero is unresolved: it is not
    # evidence that the two arms are equal, and it is never reported as a tie.
    table["verdict"] = np.where(table["ci_low"] > 0, "ahead",
                                np.where(table["ci_high"] < 0, "behind", "unresolved"))
    print(f"Paired intervals on {comparison['metric']}, grouped by the "
          f"{comparison['grouped_by']} side, against {comparison['reference']}")
    display(table[["comparison", "difference", "ci_low", "ci_high", "verdict",
                   "rows", "groups"]].set_index("comparison"))
    print(table["verdict"].value_counts().to_string())

In [ ]:
if comparison is not None:
    table = pd.DataFrame(comparison["rows"])
    mid, low, high = table["difference"], table["ci_low"], table["ci_high"]
    fig, axis = plt.subplots(figsize=(7.5, 0.55 * len(table) + 1.6))
    positions = np.arange(len(table))
    axis.errorbar(mid, positions, xerr=[mid - low, high - mid], fmt="o", color="#222222",
                  capsize=3)
    axis.axvline(0, color="#b00020", linestyle="--")
    axis.set_yticks(positions)
    axis.set_yticklabels(table["comparison"])
    axis.invert_yaxis()
    axis.set_xlabel(f"difference in {comparison['metric']} against {comparison['reference']}")
    axis.set_title(f"{frame.name}, {HELD_OUT} held out: an interval crossing the line is unresolved")
    fig.tight_layout()
    plt.show()

## What this notebook does not establish

A result here is one seed on one table under one split. Before any of it is quoted:

* **Three seeds.** The stability cohort needed three seeds to separate a real gain from
  run-to-run spread, and four of its candidate gains did not survive that. There is no reason
  to think a pair task is steadier.
* **Identity clustering.** Groups are exact-identity only, so a protein 95% identical to a
  training protein is counted as unseen. Every number here is optimistic until the entities
  are clustered at a stated identity threshold and the split is recomputed.
* **A ligand table's floor.** `davis` is 71% censored at pKd 5.0. A Spearman on it is mostly
  the model separating the censoring limit from the measurements, which is a different claim
  from ranking affinities.
* **One encoder per side.** Whether the result depends on the encoder is an encoder-swap
  question, and the stability cohort found the encoder mattered more than the fusion operation
  did.